In [9]:
# Fill in student ID and name
# 
student_id = "s225009113"
student_first_last_name = "Devindi Rajapaksha Vithanage"
print(student_id, student_first_last_name)

s225009113 Devindi Rajapaksha Vithanage


In [10]:
# install plotly and dash, if not yet already
! pip install plotly dash

import plotly, dash
print(plotly.__version__)
print(dash.__version__)

5.24.1
3.2.0


In [11]:
from dash import Dash, html

"""
Initialize the app.
 
This line is known as the Dash constructor and is responsible for initializing your app. 
It is almost always the same for any Dash app you create.
""" 
app = Dash()

"""
App layout.

The app layout represents the app components that will be displayed in the web browser and 
here is provided as a Dash component (html.Div) containing other Divs.
"""
app.layout = html.Div([
    html.Div(children='Hello World'),
    html.Div(children='My name is Devindi Chathunika Amarasinghe Rajapaksha Vithanage'),
    html.Div(children='I am learning Dash now!')   # 👈 Added another Div
])

if __name__ == '__main__':
    app.run(debug=True, port=8051, jupyter_mode="tab")  # 👈 Use a different port to avoid "already in use" error


Dash app running on http://127.0.0.1:8051/


<IPython.core.display.Javascript object>

In [19]:
# Import packages
from dash import Dash, html, dcc, Input, Output
import pandas as pd
import plotly.express as px

# Load gyroscope data
df = pd.read_csv("gyroscope_data.csv")   # Columns: timestamp, gx, gy, gz

# Initialize the app
app = Dash()

# Layout
app.layout = html.Div([
    html.H1("Gyroscope Data Dashboard", style={"textAlign": "center"}),

    # Dropdown to select chart type
    dcc.Dropdown(
        id="chart-type",
        options=[
            {"label": "Line Chart", "value": "line"},
            {"label": "Scatter Plot", "value": "scatter"},
            {"label": "Distribution Plot (Histogram)", "value": "hist"}
        ],
        value="line",   # default
        clearable=False
    ),

    # Graph container
    dcc.Graph(id="gyro-graph")
])

# Callback to update graph based on dropdown
@app.callback(
    Output("gyro-graph", "figure"),
    Input("chart-type", "value")
)
def update_graph(chart_type):
    if chart_type == "line":
        # Line chart of gx, gy, gz over time
        fig = px.line(df, x="timestamp", y=["gx", "gy", "gz"],
                      labels={"value": "Rotation (deg/s)", "timestamp": "Time"},
                      title="Line Chart of Gyroscope Data")
    elif chart_type == "scatter":
        # Scatter plot of gx vs gy, colored by gz
        fig = px.scatter(df, x="gx", y="gy", color="gz",
                         labels={"gx": "X-axis", "gy": "Y-axis", "gz": "Z-axis"},
                         title="Scatter Plot of Gyroscope Data")
    else:  # histogram
        # Histogram distribution of gx, gy, gz
        fig = px.histogram(df.melt(value_vars=["gx", "gy", "gz"], 
                                   var_name="Axis", value_name="Value"),
                           x="Value", color="Axis", barmode="overlay",
                           title="Distribution of Gyroscope Values")

    return fig


if __name__ == '__main__':
    app.run(debug=True, port=8051, jupyter_mode="tab")



Dash app running on http://127.0.0.1:8051/


<IPython.core.display.Javascript object>

In [20]:
# Import packages
from dash import Dash, html, dcc, Input, Output
import pandas as pd
import plotly.express as px

# Load gyroscope data
df = pd.read_csv("gyroscope_data.csv")   # Columns: timestamp, gx, gy, gz

# Initialize the app
app = Dash(__name__)

# Layout
app.layout = html.Div([
    html.H1(
        "Gyroscope Data Dashboard", 
        style={"textAlign": "center", "color": "#333", "marginBottom": "30px"}
    ),

    # Dropdown to select chart type
    html.Div([
        html.Label("Select Chart Type:", style={"fontWeight": "bold"}),
        dcc.Dropdown(
            id="chart-type",
            options=[
                {"label": "Line Chart", "value": "line"},
                {"label": "Scatter Plot", "value": "scatter"},
                {"label": "Distribution Plot (Histogram)", "value": "hist"}
            ],
            value="line",   # default
            clearable=False,
            style={"width": "50%"}
        )
    ], style={"textAlign": "center", "marginBottom": "20px"}),

    # Dropdown to select axes (gx, gy, gz)
    html.Div([
        html.Label("Select Gyroscope Axes:", style={"fontWeight": "bold"}),
        dcc.Dropdown(
            id="axis-select",
            options=[
                {"label": "gx", "value": "gx"},
                {"label": "gy", "value": "gy"},
                {"label": "gz", "value": "gz"}
            ],
            value=["gx", "gy", "gz"],  # default: all
            multi=True,
            style={"width": "50%"}
        )
    ], style={"textAlign": "center", "marginBottom": "20px"}),

    # Graph container
    dcc.Graph(id="gyro-graph")
])

# Callback to update graph based on chart type and selected axes
@app.callback(
    Output("gyro-graph", "figure"),
    Input("chart-type", "value"),
    Input("axis-select", "value")
)
def update_graph(chart_type, selected_axes):
    if not selected_axes:
        # If no axis selected, show empty figure
        return px.line(title="No axes selected")

    if chart_type == "line":
        fig = px.line(
            df, x="timestamp", y=selected_axes,
            labels={"value": "Rotation (deg/s)", "timestamp": "Time"},
            title="Line Chart of Gyroscope Data"
        )
    elif chart_type == "scatter":
        # For scatter, only use first two selected axes (if available)
        x_axis = selected_axes[0] if len(selected_axes) > 0 else "gx"
        y_axis = selected_axes[1] if len(selected_axes) > 1 else x_axis
        color_axis = selected_axes[2] if len(selected_axes) > 2 else None
        fig = px.scatter(
            df, x=x_axis, y=y_axis, color=color_axis,
            labels={"gx": "X-axis", "gy": "Y-axis", "gz": "Z-axis"},
            title="Scatter Plot of Gyroscope Data"
        )
    else:  # histogram
        fig = px.histogram(
            df.melt(value_vars=selected_axes, var_name="Axis", value_name="Value"),
            x="Value", color="Axis", barmode="overlay",
            title="Distribution of Gyroscope Values"
        )

    fig.update_layout(template="plotly_white")
    return fig

# Run app
if __name__ == "__main__":
    app.run(debug=True, port=8051, use_reloader=False)


In [22]:
# Import packages
from dash import Dash, html, dcc, Input, Output, State
import pandas as pd
import plotly.express as px

# Load gyroscope data
df = pd.read_csv("gyroscope_data.csv")  # Columns: timestamp, gx, gy, gz
total_samples = len(df)

# Initialize the app
app = Dash(__name__)

# Layout
app.layout = html.Div([
    html.H1(
        "Gyroscope Data Dashboard",
        style={"textAlign": "center", "color": "#333", "marginBottom": "30px"}
    ),

    # Chart type dropdown
    html.Div([
        html.Label("Select Chart Type:", style={"fontWeight": "bold"}),
        dcc.Dropdown(
            id="chart-type",
            options=[
                {"label": "Line Chart", "value": "line"},
                {"label": "Scatter Plot", "value": "scatter"},
                {"label": "Distribution Plot (Histogram)", "value": "hist"}
            ],
            value="line",
            clearable=False,
            style={"width": "50%"}
        )
    ], style={"textAlign": "center", "marginBottom": "20px"}),

    # Axis selection
    html.Div([
        html.Label("Select Gyroscope Axes:", style={"fontWeight": "bold"}),
        dcc.Dropdown(
            id="axis-select",
            options=[
                {"label": "gx", "value": "gx"},
                {"label": "gy", "value": "gy"},
                {"label": "gz", "value": "gz"}
            ],
            value=["gx", "gy", "gz"],
            multi=True,
            style={"width": "50%"}
        )
    ], style={"textAlign": "center", "marginBottom": "20px"}),

    # Number of samples input
    html.Div([
        html.Label("Number of samples to display:", style={"fontWeight": "bold"}),
        dcc.Input(
            id="num-samples",
            type="number",
            value=100,  # default number of samples
            min=1,
            max=total_samples,
            step=1,
            style={"marginRight": "20px"}
        ),
        html.Button("Previous", id="prev-btn", n_clicks=0, style={"marginRight": "10px"}),
        html.Button("Next", id="next-btn", n_clicks=0)
    ], style={"textAlign": "center", "marginBottom": "20px"}),

    # Hidden store to keep track of the current start index
    dcc.Store(id="start-index", data=0),

    # Graph
    dcc.Graph(id="gyro-graph")
])

# Callback to update the start index for navigation
@app.callback(
    Output("start-index", "data"),
    Input("next-btn", "n_clicks"),
    Input("prev-btn", "n_clicks"),
    State("num-samples", "value"),
    State("start-index", "data")
)
def update_start_index(next_clicks, prev_clicks, n_samples, start_idx):
    ctx = dash.callback_context

    if not ctx.triggered:
        return start_idx

    button_id = ctx.triggered[0]["prop_id"].split(".")[0]

    if button_id == "next-btn":
        new_idx = start_idx + n_samples
        if new_idx >= total_samples:
            new_idx = total_samples - n_samples if total_samples - n_samples >= 0 else 0
    elif button_id == "prev-btn":
        new_idx = start_idx - n_samples
        if new_idx < 0:
            new_idx = 0
    else:
        new_idx = start_idx

    return new_idx

# Callback to update graph
@app.callback(
    Output("gyro-graph", "figure"),
    Input("chart-type", "value"),
    Input("axis-select", "value"),
    Input("num-samples", "value"),
    Input("start-index", "data")
)
def update_graph(chart_type, selected_axes, n_samples, start_idx):
    if not selected_axes or n_samples <= 0:
        return px.line(title="No data to display")

    # Get the subset of data
    end_idx = start_idx + n_samples
    df_subset = df.iloc[start_idx:end_idx]

    if chart_type == "line":
        fig = px.line(
            df_subset, x="timestamp", y=selected_axes,
            labels={"value": "Rotation (deg/s)", "timestamp": "Time"},
            title=f"Line Chart of Gyroscope Data (Samples {start_idx} to {end_idx})"
        )
    elif chart_type == "scatter":
        x_axis = selected_axes[0] if len(selected_axes) > 0 else "gx"
        y_axis = selected_axes[1] if len(selected_axes) > 1 else x_axis
        color_axis = selected_axes[2] if len(selected_axes) > 2 else None
        fig = px.scatter(
            df_subset, x=x_axis, y=y_axis, color=color_axis,
            labels={"gx": "X-axis", "gy": "Y-axis", "gz": "Z-axis"},
            title=f"Scatter Plot of Gyroscope Data (Samples {start_idx} to {end_idx})"
        )
    else:  # histogram
        fig = px.histogram(
            df_subset.melt(value_vars=selected_axes, var_name="Axis", value_name="Value"),
            x="Value", color="Axis", barmode="overlay",
            title=f"Distribution of Gyroscope Values (Samples {start_idx} to {end_idx})"
        )

    fig.update_layout(template="plotly_white")
    return fig

# Run app
if __name__ == "__main__":
    app.run(debug=True, port=8051, use_reloader=False)


In [23]:
# Import packages
from dash import Dash, html, dcc, Input, Output, State, dash_table
import pandas as pd
import plotly.express as px


# Load gyroscope data
df = pd.read_csv("gyroscope_data.csv")  # Columns: timestamp, gx, gy, gz
total_samples = len(df)

# Initialize the app
app = Dash(__name__)

# Layout
app.layout = html.Div([
    html.H1(
        "Gyroscope Data Dashboard",
        style={"textAlign": "center", "color": "#333", "marginBottom": "30px"}
    ),

    # Chart type dropdown
    html.Div([
        html.Label("Select Chart Type:", style={"fontWeight": "bold"}),
        dcc.Dropdown(
            id="chart-type",
            options=[
                {"label": "Line Chart", "value": "line"},
                {"label": "Scatter Plot", "value": "scatter"},
                {"label": "Distribution Plot (Histogram)", "value": "hist"}
            ],
            value="line",
            clearable=False,
            style={"width": "50%"}
        )
    ], style={"textAlign": "center", "marginBottom": "20px"}),

    # Axis selection
    html.Div([
        html.Label("Select Gyroscope Axes:", style={"fontWeight": "bold"}),
        dcc.Dropdown(
            id="axis-select",
            options=[
                {"label": "gx", "value": "gx"},
                {"label": "gy", "value": "gy"},
                {"label": "gz", "value": "gz"}
            ],
            value=["gx", "gy", "gz"],
            multi=True,
            style={"width": "50%"}
        )
    ], style={"textAlign": "center", "marginBottom": "20px"}),

    # Number of samples input with navigation
    html.Div([
        html.Label("Number of samples to display:", style={"fontWeight": "bold"}),
        dcc.Input(
            id="num-samples",
            type="number",
            value=100,
            min=1,
            max=total_samples,
            step=1,
            style={"marginRight": "20px"}
        ),
        html.Button("Previous", id="prev-btn", n_clicks=0, style={"marginRight": "10px"}),
        html.Button("Next", id="next-btn", n_clicks=0)
    ], style={"textAlign": "center", "marginBottom": "20px"}),

    # Hidden store to keep track of the current start index
    dcc.Store(id="start-index", data=0),

    # Graph
    dcc.Graph(id="gyro-graph"),

    # Summary table
    html.H3("Summary Statistics of Displayed Data", style={"textAlign": "center", "marginTop": "30px"}),
    dash_table.DataTable(
        id="summary-table",
        style_table={"width": "70%", "margin": "0 auto"},
        style_cell={"textAlign": "center"},
        columns=[{"name": "Statistic", "id": "Statistic"}]  # Will update dynamically
    )
])

# Callback to update start index for navigation
@app.callback(
    Output("start-index", "data"),
    Input("next-btn", "n_clicks"),
    Input("prev-btn", "n_clicks"),
    State("num-samples", "value"),
    State("start-index", "data")
)
def update_start_index(next_clicks, prev_clicks, n_samples, start_idx):
    ctx = dash.callback_context
    if not ctx.triggered:
        return start_idx
    button_id = ctx.triggered[0]["prop_id"].split(".")[0]

    if button_id == "next-btn":
        new_idx = start_idx + n_samples
        if new_idx >= total_samples:
            new_idx = max(total_samples - n_samples, 0)
    elif button_id == "prev-btn":
        new_idx = max(start_idx - n_samples, 0)
    else:
        new_idx = start_idx
    return new_idx

# Callback to update graph and summary table
@app.callback(
    Output("gyro-graph", "figure"),
    Output("summary-table", "data"),
    Output("summary-table", "columns"),
    Input("chart-type", "value"),
    Input("axis-select", "value"),
    Input("num-samples", "value"),
    Input("start-index", "data")
)
def update_graph_and_table(chart_type, selected_axes, n_samples, start_idx):
    if not selected_axes or n_samples <= 0:
        return px.line(title="No data to display"), [], [{"name": "Statistic", "id": "Statistic"}]

    # Subset of data
    end_idx = start_idx + n_samples
    df_subset = df.iloc[start_idx:end_idx][selected_axes]

    # Create graph
    if chart_type == "line":
        fig = px.line(df.iloc[start_idx:end_idx], x="timestamp", y=selected_axes,
                      labels={"value": "Rotation (deg/s)", "timestamp": "Time"},
                      title=f"Line Chart of Gyroscope Data (Samples {start_idx} to {end_idx})")
    elif chart_type == "scatter":
        x_axis = selected_axes[0]
        y_axis = selected_axes[1] if len(selected_axes) > 1 else x_axis
        color_axis = selected_axes[2] if len(selected_axes) > 2 else None
        fig = px.scatter(df_subset, x=x_axis, y=y_axis, color=color_axis,
                         labels={"gx": "X-axis", "gy": "Y-axis", "gz": "Z-axis"},
                         title=f"Scatter Plot of Gyroscope Data (Samples {start_idx} to {end_idx})")
    else:
        fig = px.histogram(df_subset.melt(var_name="Axis", value_name="Value"),
                           x="Value", color="Axis", barmode="overlay",
                           title=f"Distribution of Gyroscope Values (Samples {start_idx} to {end_idx})")

    fig.update_layout(template="plotly_white")

    # Create summary table
    summary_df = df_subset.describe().T.reset_index().rename(columns={"index": "Axis"})
    summary_df = summary_df[["Axis", "mean", "std", "min", "25%", "50%", "75%", "max"]]
    columns=[{"name": col, "id": col} for col in summary_df.columns]

    return fig, summary_df.to_dict('records'), columns

# Run app
if __name__ == "__main__":
    app.run(debug=True, port=8051, use_reloader=False)


### ✅ New Features Added:

- **`dash_table.DataTable`** shows *mean*, *std*, *min*, *25%*, *50%*, *75%*, *max* for each selected axis.

- The table **updates dynamically** whenever the graph subset changes (*axis selection*, *sample count*, *navigation*).

- Works with **all chart types** and reflects the *currently displayed data*.
